# Total Care Traffic — 4단계 최소 실행 가능성 검증: 검출 + 추적 + 중심좌표 저장 + 5초 영상 (Colab + Drive)

- **목적:** 2022년 프레임 독립 검출 기준선(`src/detect.py`)의 미완성 영역이던 **시간축 객체 연결(Tracking ID) · 중심좌표/궤적 저장 · `≥5초` 추적 영상 · GT TrackEval 포맷 출력**을 검증합니다.
- **주의:** 모델 아키텍처(예: YOLOv8s/YOLOv9 등)와 Tracker(ByteTrack/BoT-SORT 등)는 실험 대상이며 사전에 고정하지 않습니다.


In [ ]:
# 1. Google Drive 마운트 및 경로 확인
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/KITTI_Tracking')
RUNS_DIR = DRIVE_ROOT / 'runs'
WORK_YOLO_DIR = Path('/content/kitti_yolo')
%cd /content/TotalCare


In [ ]:
# 2. 최소 실행 가능성 학습 (작은 epoch/검증 구간으로 3클래스 BBox 학습 가능 여부 먼저 확인)
from ultralytics import YOLO

MODEL_CKPT = 'yolov8s.pt'  # 실험할 가중치/아키텍처 선택
model = YOLO(MODEL_CKPT)
results = model.train(
    data=str(WORK_YOLO_DIR / 'data_kitti.yaml'),
    epochs=20,
    imgsz=640,
    batch=16,
    project=str(RUNS_DIR),
    name='kitti_feasibility',
    exist_ok=True,
)


In [ ]:
# 3. 검증 시퀀스 연속 프레임 추출 후 Tracking + 중심좌표 CSV + 5초 MP4 + KITTI 18-col 출력
import shutil, zipfile

DEMO_SEQ = '0015'  # >=50프레임(>=5초 @ 10 Hz) 연속 시퀀스 선택
IMG_ZIP = DRIVE_ROOT / 'data_tracking_image_2.zip'
SEQ_FRAMES_DIR = Path(f'/content/kitti_seq_{DEMO_SEQ}')
if SEQ_FRAMES_DIR.exists():
    shutil.rmtree(SEQ_FRAMES_DIR)
SEQ_FRAMES_DIR.mkdir(parents=True)

with zipfile.ZipFile(IMG_ZIP, 'r') as zf:
    for entry in sorted(zf.namelist()):
        if f'image_02/{DEMO_SEQ}/' in entry and entry.endswith('.png'):
            (SEQ_FRAMES_DIR / Path(entry).name).write_bytes(zf.read(entry))

BEST_WEIGHTS = RUNS_DIR / 'kitti_feasibility' / 'weights' / 'best.pt'
TRACK_OUT = RUNS_DIR / f'kitti_track_{DEMO_SEQ}'
!python3 src/track_utils.py \
  --weights "{BEST_WEIGHTS}" \
  --source "{SEQ_FRAMES_DIR}" \
  --output-dir "{TRACK_OUT}" \
  --tracker bytetrack.yaml \
  --fps 10 \
  --kitti-seq "{DEMO_SEQ}" 


In [ ]:
# 4. 생성된 궤적 CSV / KITTI 포맷 txt / 5초 이상 영상 확인
import pandas as pd
df = pd.read_csv(TRACK_OUT / 'trajectories.csv')
print("Records:", len(df), "| Unique Track IDs in clip:", df['track_id'].nunique())
print(df.head(10))
!head -n 5 "{TRACK_OUT / 'kitti_preds' / (DEMO_SEQ + '.txt')}" 
